In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from regpy.vecsps import NumPyVectorSpace
from regpy.operators import Operator, VectorOfOperators, Identity
from phaseless_passive_ip_ops import Mat,  Tau, MatrixAutoProductOp, DiagMatrixAutoProductOp 
from low_rank_op_misfit_fct import HilbertSchmidtLowRank
from auxiliary_ops import Reshape

In [ ]:
N=100000    # sample size
row = 5     # size of random Gaussian vectors
col = 2     # rank of covariance matrix (if smaller than row)

V = np.random.randint(low=1,high=10,size =(5,2)) + complex(0,1)*np.random.randint(low=1,high=10,size =(5,2)) - complex(5,5)

cov_phase = np.zeros((row,row),dtype=complex)
intensities = np.zeros(shape=(N,row),dtype=float)
cox_data = np.zeros(shape=(N,row),dtype=float)
for i in range(N):
    src = (np.random.randn(col) + complex(0,1)*np.random.randn(col))/np.sqrt(2)
    field = V @ src
    cov_phase += field[(...,None)]*np.conj(field)
    intensities[i] = np.abs(field)**2
    cox_data[i] = np.random.poisson(intensities[i])
cov_phase /=N
cov_exact = V@V.T.conj()

### Show elementwise relative errors of empirical covariance matrix of phased field

In [ ]:
err = np.abs(cov_exact-cov_phase)/(np.abs(cov_exact))
plt.imshow(err,cmap='hot')
plt.colorbar()
plt.title('rel. errors in empirical phased covariances')



### Compute empirical covariances of intensities and of the corresponding Cox process

In [ ]:
mean_intensity = np.sum(intensities,axis=0)/N
mean_cox = np.sum(cox_data,axis=0)/N
int_corr = np.zeros(shape = (row,row),dtype=float)
for int in intensities:
    centered_int = int-mean_intensity
    int_corr += centered_int[(...,None)]*centered_int
int_corr /=N
mean_cox = np.sum(cox_data,axis=0)/N
cox_corr = np.zeros_like(int_corr)
for int in cox_data:
    centered_int = int-mean_cox
    cox_corr += centered_int[(...,None)]*centered_int
cox_corr /=N

### Show relative errors of empirical covariance of intensity correlations of squared Gaussian fields 

In [ ]:
int_ex_cov = np.abs(cov_exact)**2 # exact covariance matrix of the intensities

err = np.abs(int_corr-int_ex_cov)/(np.abs(int_ex_cov))
plt.imshow(err,cmap='hot')
plt.title('rel. errors of empirical intensity correlations')
plt.colorbar()

### Show relative errors of empirical Cox process correlations and comparison to covariance of mean intensities. 
(The latter indicates the relevance of the diagonal term!)

In [ ]:
cox_cov = int_ex_cov + np.diag(np.sqrt(np.diag(int_ex_cov)))  #  exact covariance of the Cox process
err = np.abs(cox_corr-cox_cov)/np.abs(cox_cov)
err2 = np.abs(cox_corr-int_ex_cov)/np.abs(int_ex_cov)
fig, (ax1,ax2) = plt.subplots(2,1)
im1=ax1.imshow(err,cmap='hot')
ax1.set_title('rel. errors empirical covariance Cox process')
plt.colorbar(im1)
im2=ax2.imshow(err2,cmap='hot')
ax2.set_title('rel. difference to covariance of intensities')
plt.colorbar(im2)
int_corr, cox_corr


### Show deviation of intensity correlations from its rank-1-approximation, corresponding to perfect spatial coherence 

In [ ]:
diag =  np.diag(int_ex_cov)
rank_one_approx = np.sqrt(diag)[(...,None)]*np.sqrt(diag)
fig, (ax1,ax2,ax3) = plt.subplots(1,3)
im1=ax1.imshow(int_ex_cov,cmap='hot')
ax1.set_title('exact intensity covariance')
plt.colorbar(im1)
im2=ax2.imshow(rank_one_approx,cmap='hot')
ax2.set_title('rank 1 approximation')
plt.colorbar(im2)
im3=ax3.imshow(int_ex_cov-rank_one_approx,cmap='hot')
ax3.set_title('difference')
plt.colorbar(im3)


### Compute backpropagations of intensity and Cox data and their expectations

In [ ]:
TauOp = Tau(domain=NumPyVectorSpace((col,),dtype=complex),codomain=NumPyVectorSpace((row,),dtype=complex))
MatMul = MatrixAutoProductOp(TauOp.codomain,ndim_col=2)
Cox_cov_op = MatMul * TauOp

DiagMatMul = DiagMatrixAutoProductOp(TauOp.domain,ndim_col=1)

tau,DTauOp = TauOp.linearize(V)
aux,DMatMul = MatMul.linearize(tau,return_adjoint_eval=True)
expect_intensity, D_DiagMatMul = DiagMatMul.linearize(V)
expect_int_backprop = DTauOp.adjoint(aux)
expect_int_backprop2, D_Cox_cov_op = Cox_cov_op.linearize(V,return_adjoint_eval=True)
expect_cox_backprop = expect_int_backprop - D_DiagMatMul.adjoint(mean_intensity)

intensity_backprop = DTauOp.adjoint(MatMul._adjoint_data(intensities-mean_intensity))
cox_backprop = DTauOp.adjoint(MatMul._adjoint_data(cox_data-mean_cox))

In [ ]:
DMatMul._adjoint_derivative(tau)
DMatMul._adjoint_eval(tau)

### Generate samples of a circular Gaussian random field with covariance matrix $VV^*$ and corresponding intensities and Cox data

In [ ]:
D_DiagMatMul.adjoint(mean_intensity)

In [ ]:
#intensity_backprop/N, expectation_backprop
err_int=np.abs(intensity_backprop-expect_int_backprop)/np.abs(expect_int_backprop)
err_cox=np.abs(cox_backprop-expect_cox_backprop)/np.abs(expect_cox_backprop)
diff_cox_int=(cox_backprop-intensity_backprop)/np.abs(intensity_backprop)
expect_diff_cox_int=(expect_cox_backprop-expect_int_backprop)/np.abs(expect_int_backprop)

fig, ax = plt.subplots(2,2)
r = 0
ax[0,0].plot(intensity_backprop[:,r].real,label='int. backprop')
ax[0,0].plot(expect_int_backprop[:,r].real,label='expect. int. backprop')
ax[0,0].plot(cox_backprop[:,r].real,label='Cox backprop')
ax[0,0].plot(expect_cox_backprop[:,r].real,label='expect. Cox backprop')
ax[0,0].legend()
ax[0,0].set_title(f'real part component {r} of V')

ax[0,1].plot(err_int[:,r].real,label='rel. error int. backprop')
ax[0,1].plot(err_cox[:,r].real,label='rel. error Cox backprop')
ax[0,1].plot(diff_cox_int[:,r].real,label='rel. difference Cox vs int. backprop')
ax[0,1].plot(expect_diff_cox_int[:,r].real,label='... with expectations')
ax[0,1].legend()
ax[0,1].set_title(f'real part component {r} V')

r=0
ax[1,0].plot(intensity_backprop[:,r].imag,label='int. backprop')
ax[1,0].plot(expect_int_backprop[:,r].imag,label='expect. int. backprop')
ax[1,0].plot(cox_backprop[:,r].imag,label='Cox backprop')
ax[1,0].plot(expect_cox_backprop[:,r].imag,label='expect. Cox backprop')
ax[1,0].legend()
ax[1,0].set_title(f'imag. part component {r} of V')

ax[1,1].plot(np.abs(err_int[:,r]),label='rel. error int. backprop')
ax[1,1].plot(np.abs(err_cox[:,r]),label='rel. error Cox backprop')
ax[1,1].plot(np.abs(diff_cox_int[:,r]),label='rel. difference Cox vs int. backprop')
ax[1,1].plot(np.abs(expect_diff_cox_int[:,r]),label='... with expectations')
ax[1,1].legend()
ax[1,1].set_title(f'complex error component {r} V')


In [ ]:
diff = cox_backprop-intensity_backprop 
exp = D_DiagMatMul.adjoint(mean_intensity)
fac = sum(diff.flatten())/sum(exp.flatten())


### Checking consistency of the two approaches for incorporating matrix multiplication of tau-matrices: (a) in the forward operator, (b) in the data fidelity term

In [ ]:
Resh=Reshape(TauOp.codomain, VectorSpace(shape=(row,col**2),dtype=complex))
Double = VectorOfOperators([Identity(Resh.codomain),Identity(Resh.codomain)])
S = HilbertSchmidtLowRank(domain=Resh.codomain+Resh.codomain,data=(intensities-mean_intensity)/np.sqrt(N))
forward_op = Double*Resh*TauOp
taus,DF = forward_op.linearize(V)
aux = S.subgradient(taus)
check_backprop = DF.adjoint(aux)
np.abs(expect_int_backprop-intensity_backprop - check_backprop)/np.abs(check_backprop)

### Checking the consistency of the different forms of the diagonal term

In [ ]:
DiagMatMul1 = DiagMatrixAutoProductOp(TauOp.domain,ndim_col=1)
DiagMatMul2 = DiagMatrixAutoProductOp(TauOp.codomain,ndim_col=2)

DiagMatMul2(TauOp(V))-DiagMatMul1(V)**2